In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, Mxfp4Config

# Only reasoning models
# model_id = "openai/gpt-oss-20b"
# model_id = "google/gemma-4-26B-A4B-it"
model_id = "unsloth/gemma-4-26B-A4B-it"

# quantization_config = Mxfp4Config(
#     load_in_4bit=True,
#     bnb_4bit_compute_dtype="float16",
#     bnb_4bit_use_double_quant=True,
#     bnb_4bit_quant_type="nf4",
# )
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype="float16",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=quantization_config, device_map="auto")

config.json:   0%|          | 0.00/3.86k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/22.1k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 32.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/18.9k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/103k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

In [2]:
from utils.utils_functions import extract_harmony_final_channel

message = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Hello, how are you?"}]

prompt = tokenizer.apply_chat_template(
    message,
    tokenize=True,
    reasoning_effort='low',
    add_generation_prompt=True,
    return_tensors="pt"
).to(model.device)

output = model.generate(**prompt, max_new_tokens=100)
response = tokenizer.decode(output[0][prompt['input_ids'].shape[1]:], skip_special_tokens=False)

print(output[0][prompt['input_ids'].shape[1]:])
print(response)
print(extract_harmony_final_channel(response))

tensor([236777, 236789, 236757,   3490,   1388, 236764,   7806,    611,    573,
         10980, 236888,   2088,    659,    611,   3490,   3124, 236881,   2375,
           993,   4658,    564,    740,   1601,    611,    607, 236881,    106],
       device='cuda:0')
I'm doing well, thank you for asking! How are you doing today? Is there anything I can help you with?<turn|>
I'm doing well, thank you for asking! How are you doing today? Is there anything I can help you with?<turn|>


In [8]:
print(tokenizer.convert_ids_to_tokens(prompt['input_ids'][0]))
print(tokenizer.convert_ids_to_tokens(output[0]))

['<bos>', '<|turn>', 'system', '\n', 'You', '▁are', '▁a', '▁helpful', '▁assistant', '.', '<turn|>', '\n', '<|turn>', 'user', '\n', 'Hello', ',', '▁how', '▁are', '▁you', '?', '<turn|>', '\n', '<|turn>', 'model', '\n', '<|channel>', 'thought', '\n', '<channel|>']
['<bos>', '<|turn>', 'system', '\n', 'You', '▁are', '▁a', '▁helpful', '▁assistant', '.', '<turn|>', '\n', '<|turn>', 'user', '\n', 'Hello', ',', '▁how', '▁are', '▁you', '?', '<turn|>', '\n', '<|turn>', 'model', '\n', '<|channel>', 'thought', '\n', '<channel|>', 'I', "'", 'm', '▁doing', '▁well', ',', '▁thank', '▁you', '▁for', '▁asking', '!', '▁How', '▁are', '▁you', '▁doing', '▁today', '?', '▁Is', '▁there', '▁anything', '▁I', '▁can', '▁help', '▁you', '▁with', '?', '<turn|>']


In [3]:
for token, token_id in zip(tokenizer.convert_ids_to_tokens(output[0][prompt['input_ids'].shape[1]:]), output[0][prompt['input_ids'].shape[1]:]):
    print(f"{token}: {token_id}")

I: 236777
': 236789
m: 236757
▁doing: 3490
▁well: 1388
,: 236764
▁thank: 7806
▁you: 611
▁for: 573
▁asking: 10980
!: 236888
▁How: 2088
▁are: 659
▁you: 611
▁doing: 3490
▁today: 3124
?: 236881
▁Is: 2375
▁there: 993
▁anything: 4658
▁I: 564
▁can: 740
▁help: 1601
▁you: 611
▁with: 607
?: 236881
<turn|>: 106
